# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a comprehensive walkthrough for loading and exploring the FAIR^2 protein data dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. The dataset follows the Croissant standard schema, supporting robust, interoperable data loading and processing.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`. This block prepares the metadata for exploration and loads the schema referenced above.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Define the Croissant dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset Name: ", metadata.name)
print("Description: ", metadata.description)
print("Version:", metadata.version)
print("Published Date:", metadata.datePublished)
print("License:", metadata.license)

## 2. Data Overview

Explore available `recordSet`s, their fields, columns, and corresponding `@id`s as referenced in the Croissant schema. The `@id` is essential for precise data access and manipulation.

In [ ]:
# List out all record sets, fields, and columns by their '@id'
print("\nAvailable record sets and their fields/columns:")
record_sets = []
for rs in metadata.record_sets:
    print(f"[recordSet @id] {rs.id}")
    record_sets.append(rs.id)
    print("  Name:", rs.name)
    print("  Fields:")
    for field in rs.fields:
        print(f"    [field @id] {field.id:40s}  name: {field.name:20s}  dataType: {field.data_type}")
        if hasattr(field, 'columns') and field.columns:
            print("      Columns:")
            for col in field.columns:
                print(f"        [column @id] {col.id:40s}  name: {col.name}")
    print("")

## 3. Data Extraction

We'll extract records from each `recordSet` (using the `@id` reference), converting them to Pandas DataFrames for analysis. Replace the specific `@id` string as needed for your exploration.

In [ ]:
# Extract data from all discovered record sets
dfs = {}
for rs_id in record_sets:
    print(f"Loading records from recordSet @id: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dfs[rs_id] = df
    print(f"  Columns: {df.columns.tolist()}")
    print(f"  Number of records: {len(df)}\n")
# For this example, pick the first available record set for EDA
main_record_set_id = record_sets[0]
print(f"Selected main record set for EDA: {main_record_set_id}")
dfs[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

We'll apply common processing steps: filtering records, normalizing numeric fields, grouping, and basic summarization. **All fields/columns are accessed by their `@id`.**

Let's
1. Pick a numeric field for filtering & normalization (e.g., peptide count or MW).
2. Filter for proteins with MW > 25000.
3. Normalize MW.
4. Group by a categorical field if present (e.g., description or class).

(Update field `@id`s as printed in the previous overview cell if they are different.)

In [ ]:
# Update these as per output from cell above
# Example:
#   MW_field_id = 'cr:MW'
#   description_field_id = 'cr:description'

# Assign field @ids (UPDATE based on overview)
MW_field_id = None
name_field_id = None
for f in dataset.metadata.record_set(main_record_set_id).fields:
    if 'weight' in (f.name or '').lower() or 'mw' in (f.name or '').lower():
        MW_field_id = f.id
    if f.name and ('description' in f.name.lower() or 'protein' in f.name.lower()):
        name_field_id = f.id
print(f'MW field @id: {MW_field_id}')
print(f'Description/Name field @id: {name_field_id}')

df = dfs[main_record_set_id]
# coerce MW to numeric, skip errors
df[MW_field_id] = pd.to_numeric(df[MW_field_id], errors='coerce')
threshold = 25000

filtered_df = df[df[MW_field_id] > threshold].copy()
print(f"Filtered records with MW ({MW_field_id}) > {threshold}, count: {len(filtered_df)}")
display_cols = [MW_field_id]
if name_field_id is not None:
    display_cols.append(name_field_id)
print(filtered_df[display_cols].head())

# Normalize MW
filtered_df[f"{MW_field_id}_normalized"] = (
    filtered_df[MW_field_id] - filtered_df[MW_field_id].mean()
) / filtered_df[MW_field_id].std()
print("\nNormalized MW:")
print(filtered_df[[MW_field_id, f"{MW_field_id}_normalized"]].head())

# Group by description/name field if present
if name_field_id is not None and name_field_id in filtered_df.columns:
    grouped = filtered_df.groupby(name_field_id)[MW_field_id].mean()
    print(f"\nGrouped by {name_field_id} (mean MW):")
    print(grouped.head())

## 5. Visualization

Visualize data distributions and relationships. We'll plot MW distribution (histogram) and MW vs. description if appropriate.

In [ ]:
# MW histogram
plt.figure(figsize=(8,5))
filtered_df[MW_field_id].plot.hist(bins=30, alpha=0.7)
plt.xlabel('Molecular Weight (MW)')
plt.title('Distribution of Protein Molecular Weight (MW) (MW > 25000)')
plt.grid(True, axis='y')
plt.show()

# Top proteins by MW
if name_field_id is not None:
    topN = 10
    top_mw = filtered_df[[name_field_id, MW_field_id]].sort_values(MW_field_id, ascending=False).head(topN)
    plt.figure(figsize=(10,4))
    plt.barh(top_mw[name_field_id], top_mw[MW_field_id], color='skyblue')
    plt.xlabel('Molecular Weight (MW)')
    plt.title(f'Top {topN} Proteins by MW')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

## 6. Conclusion

- Successfully loaded and explored a Croissant-compliant FAIR^2 mass spec protein dataset using `mlcroissant`.
- Inspected all record sets, fields, and columns via their `@id`s for reproducible access.
- Filtered and analyzed records by molecular weight, including normalization and grouping/categorization by protein field.
- Visualizations revealed the MW distribution and highlighted the top candidate proteins in terms of mass.

This provides a robust foundation for further proteomics data mining and machine learning workflows. For further analysis, consult the dataset schema or field descriptions when applying advanced filtering or feature engineering steps.

_Template and methodology based on [mlcroissant](https://github.com/mlcommons/croissant) best practices._